# 04b · 나이, 잔여수명, 검사 역설 (Age, Residual Life, and the Inspection Paradox)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §3.3 (Age and Residual Life; 3.3.1 이산, 3.3.2 일반), §3.1 (재생-보상 재사용) · Ross 12e §7.7 (Inspection Paradox), §7.3, §5.3 · G&S 4e §10.3 (excess life, 평형분포), §10.2 (key renewal theorem) · Lawler 2e Ch. 6 |
| 선수 노트북 | 04a (재생 정리·재생-보상), 03e (PASTA), 02d (푸아송 나이·잔여수명), 02b, 00c |
| 예상 소요 | 90분 |
| 상태 | draft |

시각 $t$에 재생 과정을 들여다보면 세 개의 길이가 보인다: 마지막 재생 이후 흐른 시간(나이), 다음 재생까지 남은 시간(잔여수명), 그리고 둘의 합(시각 $t$를 덮는 간격의 길이). 이 노트북의 질문은 하나다 — **무작위 시각에 관찰한 간격은 왜 "보통" 간격보다 긴가?** 답은 재생-보상 정리 한 줄로 나오고, 그 답이 곧 검사 역설(inspection paradox)이다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.renewal import renewal_times, age_residual_process
from spkit.poisson import poisson_arrivals, age_residual_at
from spkit.plots import plot_hist_vs_pdf

SEED, rng = rng_for("04b")
setup_plots()
N_PATHS = scale(40_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답해 보세요. 답은 접힌 블록에 있습니다.

**Q1.** (04a) 재생 과정에서 $N(t)/t$의 극한과 그 증명의 핵심 부등식은? Erlang(2,1) 간격이면 값은?

<details><summary>정답</summary>

$N(t)/t\to 1/\mu$ a.s. 핵심은 샌드위치 $S_{N(t)}\le t<S_{N(t)+1}$을 $N(t)$로 나누고 강대수법칙을 쓰는 것. Erlang(2,1)은 $\mu=2$이므로 극한은 $1/2$.

</details>

**Q2.** (04a) 재생-보상 정리의 결론을 쓰고, 보상 $R_i$가 주기 길이 $X_i$에 의존해도 되는지 답하라.

<details><summary>정답</summary>

$R(t)/t\to E[R]/E[X]$ a.s. (기댓값 버전 $E[R(t)]/t\to E[R]/E[X]$도 성립). 쌍 $(X_i,R_i)$가 주기끼리 i.i.d.이기만 하면 한 주기 안에서 $R_i$가 $X_i$에 의존하는 것은 허용된다. 이 노트북은 바로 그 자유를 쓴다: $R_i=\min(X_i,x)$, $X_i^2/2$, $X_i^2$.

</details>

**Q3.** (04a) 수명이 지수분포일 때 수명 교체(age replacement)의 최적 $T^*$는? 이유는?

<details><summary>정답</summary>

$T^*=\infty$, 즉 예방 교체를 하지 않는다. 무기억성 때문에 헌 부품과 새 부품의 잔여수명 분포가 같아 예방 교체는 비용만 든다. 최적 조건식의 좌변 $g(T)$가 항등적으로 0이 된다.

</details>

**Q4.** (03e) M/M/c/K에서 도착 고객이 손실될 확률은 정상분포의 어느 성분이며, 그렇게 말할 수 있는 근거는?

<details><summary>정답</summary>

$\pi_K$. 근거는 PASTA: 푸아송 도착이 보는 시스템 상태의 분포는 시간평균 정상분포와 같다. 이 노트북의 검사 역설은 그 반대편 이야기다 — "무작위 시각에 본 것"이 왜 개수평균이 아니라 시간평균(그것도 길이에 비례해 가중된)인지.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

재생 시각 $S_n=X_1+\dots+X_n$, 간격 $X_i\overset{iid}\sim F$, $\mu=E[X]<\infty$, $\bar F=1-F$, $N(t)=\max\{n:S_n\le t\}$.

- **나이(age, current life)** $A(t)=t-S_{N(t)}$: 마지막 재생 이후 흐른 시간.
- **잔여수명(residual life, excess life)** $R(t)=S_{N(t)+1}-t$: 다음 재생까지 남은 시간.
- **관찰 구간 길이(spread, total life)** $L(t)=A(t)+R(t)=X_{N(t)+1}$: 시각 $t$를 덮는 간격의 길이.
- **평형분포(equilibrium distribution)** $F_e(x)=\dfrac1\mu\displaystyle\int_0^x\bar F(y)\,dy$, 밀도 $f_e(x)=\bar F(x)/\mu$. 평균은 $\dfrac{E[X^2]}{2\mu}=\dfrac\mu2(1+\mathrm{CV}^2)$, $\mathrm{CV}^2=\sigma^2/\mu^2$.
- **길이 편향(length-biased, size-biased) 분포**: 밀도 $x f(x)/\mu$. 평균 $E[X^2]/\mu$.
- **격자(lattice) 분포**: $X$의 값이 $\{0,d,2d,\dots\}$ 안에 있는 경우(상수 간격이 대표). 아니면 **비격자(non-lattice)**.

**작업 예제 Gamma(2,1)** (= Erlang(2,1)): $f(x)=xe^{-x}$, $\bar F(x)=(1+x)e^{-x}$, $\mu=2$, $E[X^2]=6$, $E[X^3]=24$. 따라서

$$f_e(x)=\frac{(1+x)e^{-x}}2,\qquad \bar F_e(x)=\Big(1+\frac x2\Big)e^{-x},\qquad \text{길이 편향 밀도 }\frac{x^2e^{-x}}2=\text{Gamma}(3,1).$$

### 2.2 정리 1 — 시간평균 버전 (재생-보상; Durrett 3e §3.3)

$\mu<\infty$이면 임의의 $x\ge0$에 대해

$$\frac1t\int_0^t\mathbf 1\{A(s)\le x\}\,ds\to F_e(x)\ \text{a.s.},\qquad \frac1t\int_0^tA(s)\,ds\to\frac{E[X^2]}{2\mu},\qquad \frac1t\int_0^tL(s)\,ds\to\frac{E[X^2]}{\mu},$$

잔여수명 $R$도 $A$와 같은 극한을 가진다. **격자 여부와 무관.**

가정이 왜 필요한가:
- $\mu<\infty$: 재생-보상 정리의 분모. $E[X^2]=\infty$이면 평균의 극한은 $\infty$이지만 정리 자체는 여전히 참이다.
- 격자 조건이 필요 없는 이유: 시간평균은 주기별 적분의 합이라 "시각 $t$가 주기의 어디에 떨어지는가"의 주기성이 평균화되어 사라진다.

<details><summary>증명</summary>

04a 정리 3(재생-보상)에 보상을 골라 넣는다. 주기 $i$ 안에서 나이는 0에서 $X_i$까지 기울기 1로 증가하므로

$$R_i=\int_{S_{i-1}}^{S_i}\mathbf 1\{A(s)\le x\}\,ds=\min(X_i,x).$$

쌍 $(X_i,R_i)$는 $X_i$만의 함수이므로 주기끼리 i.i.d.이고 $0\le R_i\le x$라 $E|R_i|<\infty$. 재생-보상 정리에 의해 시간평균은 $E[\min(X,x)]/\mu$로 수렴한다. 꼬리 적분 공식으로 $E[\min(X,x)]=\int_0^x P(\min(X,x)>y)\,dy=\int_0^x\bar F(y)\,dy$이므로 극한은 $\frac1\mu\int_0^x\bar F=F_e(x)$.

보상이 음이 아니므로 $\frac1t\sum_{i\le N(t)}R_i\le\frac1t\int_0^t\dots\le\frac1t\sum_{i\le N(t)+1}R_i$이고, 미완료 주기의 기여는 04a와 같은 샌드위치 논법으로 무시된다(양 끝이 같은 극한).

평균: $R_i=\int_0^{X_i}s\,ds=X_i^2/2$, $E[X^2]<\infty$를 가정하면 극한은 $E[X^2]/(2\mu)$. 잔여수명은 주기 안에서 $X_i$에서 0으로 감소하므로 $\int_0^{X_i}(X_i-s)\,ds=X_i^2/2$, 같은 값; 분포 버전도 $\int_0^{X_i}\mathbf 1\{X_i-s\le x\}ds=\min(X_i,x)$로 같다. $L$: 주기 내내 $L\equiv X_i$이므로 $R_i=X_i\cdot X_i=X_i^2$, 극한 $E[X^2]/\mu$. $\square$

</details>

### 2.3 정리 2 — 극한분포 (비격자; key renewal theorem)

$F$가 비격자이고 $\mu<\infty$이면

$$P(A(t)\le x)\to F_e(x),\qquad P(R(t)\le x)\to F_e(x),\qquad P(A(t)>x,\,R(t)>y)\to\bar F_e(x+y)=\frac1\mu\int_{x+y}^\infty\bar F(u)\,du.$$

($F$가 밀도 $f$를 가지면) 극한 결합밀도는 $f(a+r)/\mu$ ($a,r\ge0$).

가정이 왜 필요한가:
- 비격자: 상수 간격 $X\equiv d$이면 $A(t)=t\bmod d$가 결정적이어서 $t\to\infty$ 극한분포가 없다(E3에서 확인). 시간평균은 여전히 $F_e=$ Uniform$(0,d)$.
- $\mu<\infty$: 그래야 $F_e$의 전체 질량이 1이다.

*증명 (인용).* 마지막 재생 시각으로 조건을 주면 재생 방정식 $P(R(t)>y)=\bar F(t+y)+\int_0^t\bar F(t-s+y)\,dm(s)$가 나오고($m$은 재생 함수), 비격자·직접 리만적분가능 피적분함수에 대한 key renewal theorem이 우변을 $\frac1\mu\int_0^\infty\bar F(u+y)\,du=\bar F_e(y)$로 보낸다. 결합분포는 같은 방정식에 $\bar F(\max(x,t-s)+y)$를 넣는다. 상세는 G&S 4e §10.2–10.3, Ross 12e §7.3, §7.7. 이 노트북은 Gamma(2,1)에서 시뮬레이션으로 확인한다.

결합밀도 $f(a+r)/\mu$에서 바로 나오는 값들: $E[AR]=\frac1\mu\int_0^\infty f(u)\frac{u^3}6du=\frac{E[X^3]}{6\mu}$, $E[A^2]=\frac{E[X^3]}{3\mu}$. Gamma(2,1): $E[AR]=2$, $E[A^2]=4$, $\mathrm{Cov}(A,R)=2-1.5^2=-1/4$, $\mathrm{Var}(A)=7/4$, $\mathrm{Corr}(A,R)=-1/7$.

### 2.4 정리 3 — 검사 역설 (inspection paradox; Ross 12e §7.7)

모든 $t$에 대해 $P(L(t)>x)\ge\bar F(x)$, 즉 $L(t)$는 확률적으로 $X_1$보다 크다. 비격자·$\mu<\infty$이면 ($F$가 밀도 $f$를 가질 때) $L(t)$의 극한밀도는 $xf(x)/\mu$이고

$$\lim_{t\to\infty}E[L(t)]=\frac{E[X^2]}{\mu}=\mu+\frac{\sigma^2}{\mu}\ \ge\ \mu,$$

등호는 $X$가 상수일 때만.

가정이 왜 필요한가:
- 첫 부등식은 조건이 없다: $S_{N(t)}=s$로 조건 주면 덮는 간격에는 "$t-s$보다 길다"는 조건이 붙고, $P(X>x\mid X>t-s)\ge P(X>x)$.
- 밀도 공식은 정리 2의 결합밀도 $f(a+r)/\mu$를 $a+r=\ell$인 선분 위에서 적분한 것 — 길이 $\ell$인 간격은 시각을 "길이에 비례해" 붙잡는다.

*증명 스케치.* $P(L(t)>x\mid S_{N(t)}=s)=P(X>x\mid X>t-s)=\bar F(\max(x,t-s))/\bar F(t-s)\ge\bar F(x)$이고, $s$에 대해 조건을 풀면 첫 주장. 극한밀도: $\int_0^\ell f(\ell)/\mu\,da=\ell f(\ell)/\mu$ (E2에서 직접 유도). $E[L]-\mu=E[X^2]/\mu-\mu=\sigma^2/\mu$. 길이 편향 밀도의 총질량은 $\int xf(x)\,dx/\mu=1$.

### 2.5 정리 4 — 푸아송 특수 경우 (02d 재확인)

$X\sim\mathrm{Exp}(\lambda)$이면 $F_e=F$ (그리고 $F_e=F$인 분포는 지수분포뿐), 모든 $t$에서 $R(t)\sim\mathrm{Exp}(\lambda)$, $A(t)\sim\min(t,\mathrm{Exp}(\lambda))$, 둘은 독립, $E[L(t)]=(2-e^{-\lambda t})/\lambda\to2/\lambda$.

가정이 왜 필요한가:
- 무기억성이 전부다. $R(t)$가 과거와 독립인 Exp가 되는 것은 지수분포의 특성이고, 다른 분포에서는 $A$와 $R$이 의존한다.

*증명 스케치.* $\bar F(x)/\mu=\lambda e^{-\lambda x}=f(x)$. 역으로 $F_e=F\iff\bar F=\mu f=-\mu\bar F'\iff\bar F(x)=e^{-x/\mu}$. $E[A(t)]=E[\min(t,X)]=(1-e^{-\lambda t})/\lambda$, $E[R(t)]=1/\lambda$, 합이 02d의 공식.

### 2.6 직관

시각 $t$를 무작위로 찍으면 긴 간격에 떨어질 확률이 그 길이에 비례한다. 버스 간격이 평균 10분이어도 "내가 정류장에 도착했을 때" 걸리는 간격은 평균보다 길다. 그래서 잔여수명의 평균은 $\mu/2$가 아니라 $\frac\mu2(1+\mathrm{CV}^2)$: 간격이 규칙적일수록($\mathrm{CV}\to0$) $\mu/2$에 가깝고, 지수분포($\mathrm{CV}=1$)면 정확히 $\mu$, 더 변동이 크면 $\mu$보다 길다.

나이와 잔여수명은 같은 극한분포 $F_e$를 가지지만(구간 안의 위치가 균등하니 대칭) 독립이 아니다. 둘의 합이 길이 편향된 $L$이므로 Gamma(2,1)에서는 $\mathrm{Cov}=-1/4$로 음의 상관. 지수분포만 예외적으로 독립.

### 2.7 함정 (traps)

- 나이와 잔여수명의 극한 주변분포는 같지만(둘 다 $F_e$) 독립이 아니다. 지수분포에서만 독립. Gamma(2,1): $\mathrm{Corr}(A,R)=-1/7$.
- $F_e=F$는 지수분포에서만. "평균 잔여수명 $=\mu/2$"는 상수 간격에서만, "$=\mu$"는 지수에서만; 일반식은 $E[X^2]/(2\mu)$.
- 격자 분포(상수 간격 등)에서는 $t\to\infty$ 극한분포가 없다. 시간평균은 항상 $F_e$. 시뮬레이션에서 $t=30$과 $t=30.5$의 나이 분포가 다르면 격자 신호.
- $E[L]=E[X^2]/\mu$는 상수 간격이 아니면 $\mu$보다 엄격히 크다. $E[X^2]=\infty$(Pareto $\alpha\le2$)이면 $\mu<\infty$여도 $E[L]=\infty$.
- 시뮬레이션에서 "간격을 지표(index)로 뽑아 평균"하면 $\mu$가, "시각으로 뽑아 덮는 간격을 평균"하면 $E[X^2]/\mu$가 나온다. 어느 쪽이 질문에 맞는지가 곧 검사 역설이다.
- 유한 $t$에서는 $A(t)\le t$이므로 $E[A(t)]<E_{F_e}[A]$. 극한값과 비교하려면 $t$가 $\mu$의 여러 배여야 한다(Gamma(2,1)은 $t=30$이면 오차가 $e^{-30}$ 수준이지만 무거운 꼬리 분포는 훨씬 느리다).

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요)

"$\mathrm{Cov}(A,R)<0$"과 "$L$이 클수록 $A,R$ 모두 큰 경향"이 모순처럼 들리는데, 왜 둘 다 참인지(조건부 vs 비조건부) 두 문장으로 정리해 보세요. 그리고 $E[L]=E[X^2]/\mu$가 왜 $\mu$보다 큰지를 "길이에 비례한 가중치"로 설명해 보세요.

## 3. Predict — 코드를 돌리기 전에 예측

돌리기 전에 숫자를 적어 보세요. 정리 1–4와 Gamma(2,1)의 닫힌 형식만으로 모두 계산할 수 있습니다.

In [ ]:
predictions = {
    # Gamma(2,1) 간격(mu=2, E[X^2]=6) 재생 과정에서 t→∞일 때 나이 A(t)의 평균은? (잔여수명 R(t)도 같은 값)
    "mean_age_limit": None,
    # 시각 t를 덮는 간격 길이 L(t)의 극한 평균은? (mu=2와 비교)
    "mean_spread_limit": None,
    # 극한에서 P(A > 2)는?
    "P_age_gt_2": None,
    # 극한에서 P(A > 1, R > 1)는? (독립이라면 Fe_bar(1)^2 = (1.5 e^{-1})^2 ≈ 0.3045가 되어야 할 것)
    "P_age_gt1_res_gt1": None,
    # 극한에서 Cov(A, R)의 부호와 값은?
    "cov_age_residual": None,
    # 비율 1 푸아송 과정에서 t=0.5를 덮는 간격 길이의 평균 E[L(0.5)]은? (t→∞ 극한은 2)
    "E_spread_poisson_t05": None,
    # 비율 1 푸아송 과정에서 t=0.5의 평균 나이 E[A(0.5)]은? (E[R(0.5)]=1과 비교)
    "E_age_poisson_t05": None,
}

## 4. Simulate — 시뮬레이션

먼저 이 노트북에서 반복해 쓰는 Gamma(2,1) 닫힌 형식과, 4만 경로를 한 번에 처리하는 벡터화 헬퍼를 정의한다. `spkit.age_residual_at`은 경로 하나용이라 여기서 다경로 버전을 만든다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def age_residual_many(sampler_shape, t, n_paths, n_cols, rng):
    """n_paths개의 경로에서 시각 t의 (나이, 잔여수명). sampler_shape(shape, rng) -> 간격 배열."""
    S = sampler_shape((n_paths, n_cols), rng).cumsum(axis=1)
    assert (S[:, -1] > t).all(), "n_cols를 늘리세요: 어떤 경로는 t에 못 미쳤다"
    idx = (S <= t).sum(1)                       # N(t) per path
    rows = np.arange(n_paths)
    last = np.where(idx > 0, S[rows, np.maximum(idx - 1, 0)], 0.0)   # S_{N(t)}
    nxt = S[rows, idx]                                              # S_{N(t)+1}
    return t - last, nxt - t

def covering_lengths(times, U):
    """재생 시각 times(정렬) 위에서 시각 U를 덮는 간격의 길이. U는 (0, times[-1]) 안."""
    gaps = np.diff(np.concatenate(([0.0], times)))
    return gaps[np.searchsorted(times, U)]

# Gamma(2,1) 전용 닫힌 형식
MU, EX2, EX3 = 2.0, 6.0, 24.0
f = lambda x: x * np.exp(-x)                       # interarrival density
Fbar = lambda x: (1 + x) * np.exp(-x)              # survival
f_e = lambda x: Fbar(x) / MU                       # equilibrium density
Fe_bar = lambda x: (1 + x / 2) * np.exp(-x)        # equilibrium survival
f_len = lambda x: x * f(x) / MU                    # length-biased density = Gamma(3,1)
Gbar_len = lambda x: (1 + x + x**2 / 2) * np.exp(-x)   # its survival
gamma_shape = lambda shape, g: g.gamma(2.0, 1.0, shape)
sampler = lambda n, g: g.gamma(2.0, 1.0, n)

### 4.1 한 경로의 톱니 (Fig 1)

한 경로에서 나이는 재생 때마다 0으로 떨어졌다가 기울기 1로 오르고, 잔여수명은 재생 직후 새 간격 길이에서 시작해 기울기 $-1$로 내려간다. 둘의 합은 각 구간에서 상수(그 간격의 길이)다.

In [ ]:
times = renewal_times(sampler, 30.0, rng)
t_grid = np.linspace(0, 30, 3001)
A1, R1 = age_residual_process(times, t_grid)
ok = np.isfinite(R1)                     # 마지막 재생 이후에는 R = inf (다음 재생을 안 뽑았으므로)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t_grid, A1, label="age A(t)")
ax.plot(t_grid[ok], R1[ok], label="residual R(t)")
ax.plot(t_grid[ok], (A1 + R1)[ok], "k:", lw=1, label="A + R = L(t)")
ax.plot(times, np.zeros_like(times), "|", color="k", ms=14, label="renewal epochs")
ax.set_xlabel("t"); ax.set_ylabel("time"); ax.set_title("Fig 1: age and residual life, one Gamma(2,1) renewal path")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.18)); plt.show()
print(f"renewals by t=30: {len(times)}  (expected about 30/mu = {30 / MU:.0f})")

### 4.2 $t=30$에서 여러 경로: 나이와 잔여수명의 분포 (Fig 2)

이제 독립 경로 `N_PATHS`개를 만들어 시각 $t=30$에서의 $(A,R)$을 한 번씩 읽는다. $t=30$은 $\mu=2$의 15배라 Gamma(2,1)에서는 극한분포와의 차이가 $e^{-30}$ 수준이다. 히스토그램에 평형밀도 $f_e$(실선)와 원래 간격 밀도 $f$(점선)를 겹친다.

In [ ]:
T_OBS, N_COLS = 30.0, 40          # 40개 간격의 합은 평균 80, sd 9 → t=30을 못 넘길 확률은 무시할 수준
A, R = age_residual_many(gamma_shape, T_OBS, N_PATHS, N_COLS, rng)
L = A + R
x = np.linspace(0, 10, 400)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, s, name in zip(axes, (A, R), ("age A(30)", "residual R(30)")):
    plot_hist_vs_pdf(s, x, f_e(x), ax=ax, label_pdf="f_e(x) = (1+x)e^{-x}/2")
    ax.plot(x, f(x), "--", color="C3", label="f(x) = x e^{-x}")
    ax.set_title(f"Fig 2: {name}, Gamma(2,1)"); ax.set_xlabel("x"); ax.set_xlim(0, 8); ax.legend()
plt.show()

In [ ]:
est_A, est_R = mc_estimate(A), mc_estimate(R)
p_A2 = mc_proportion(A > 2)
p_A1R1 = mc_proportion((A > 1) & (R > 1))
print(f"E[A(30)] = {est_A}   (limit E[X^2]/(2 mu) = {EX2 / (2 * MU)})")
print(f"E[R(30)] = {est_R}")
print(f"P(A > 2)      = {p_A2}   (Fe_bar(2) = {Fe_bar(2.0):.4f})")
print(f"P(A>1, R>1)   = {p_A1R1}   (Fe_bar(2) = {Fe_bar(2.0):.4f}; if independent: {Fe_bar(1.0)**2:.4f})")

$P(A>1,R>1)$이 $\bar F_e(1)^2$가 아니라 $\bar F_e(2)$ 쪽에 붙는다 — 주변분포가 같아도 독립이 아니다.

### 4.3 관찰 구간 길이 $L=A+R$ (Fig 3)

같은 표본에서 $L=A+R$의 분포를 본다. 예측은 길이 편향 밀도 $xf(x)/\mu=x^2e^{-x}/2$, 즉 Gamma(3,1)이다. 원래 간격 밀도 $f$(점선)와 비교하면 질량이 오른쪽으로 밀려 있다.

In [ ]:
ax = plot_hist_vs_pdf(L, x, f_len(x), label_pdf="x f(x)/mu = x^2 e^{-x}/2  (Gamma(3,1))")
ax.plot(x, f(x), "--", color="C3", label="f(x) = x e^{-x}  (mean 2)")
ax.axvline(L.mean(), color="k", lw=1, ls=":", label=f"sample mean {L.mean():.2f}")
ax.set_title("Fig 3: spread L(30) = A + R, Gamma(2,1)"); ax.set_xlabel("x"); ax.set_xlim(0, 12); ax.legend()
plt.show()

est_L = mc_estimate(L)
print(f"E[L(30)]   = {est_L}   (limit E[X^2]/mu = {EX2 / MU})")
print(f"E[L(30)^2] = {mc_estimate(L**2)}   (limit E[X^3]/mu = {EX3 / MU})")
for xx in (2.0, 4.0):
    print(f"P(L > {xx:.0f}) = {mc_proportion(L > xx)}   >= Fbar({xx:.0f}) = {Fbar(xx):.3f};  limit Gbar_len({xx:.0f}) = {Gbar_len(xx):.3f}")

정리 3의 부등식 $P(L(t)>x)\ge\bar F(x)$가 $x=2,4$에서 넉넉히 성립하고, 추정치는 극한값 $\bar G(x)=(1+x+x^2/2)e^{-x}$ 쪽에 붙는다.

### 4.4 길이 편향 표본추출을 한 경로에서 (Fig 4)

검사 역설의 기계적 본질은 **표본추출 방식**이다. 한 긴 경로에서 (i) 간격을 지표로 뽑아 평균하면 $\mu=2$, (ii) $[0,T]$의 균등 무작위 시각 $U$를 덮는 간격을 평균하면 $E[X^2]/\mu=3$이 나온다. 같은 경로라 두 표본이 독립이 아니므로 5절의 compare 표에는 넣지 않고 그림과 숫자만 본다.

In [ ]:
T_PATH, M = scale(100_000), scale(20_000)
times_long = renewal_times(sampler, T_PATH, rng)
gaps = np.diff(np.concatenate(([0.0], times_long)))          # sampled by index
U = rng.uniform(0, times_long[-1], M)
L_U = covering_lengths(times_long, U)                        # sampled by time

bins = np.linspace(0, 12, 61)
fig, ax = plt.subplots()
ax.hist(gaps, bins=bins, density=True, alpha=0.5, label=f"by index: mean {gaps.mean():.3f}")
ax.hist(L_U, bins=bins, density=True, alpha=0.5, label=f"by time: mean {L_U.mean():.3f}")
ax.plot(x, f(x), color="C0", lw=2, label="x e^{-x}  (mean 2)")
ax.plot(x, f_len(x), color="C1", lw=2, label="x^2 e^{-x}/2  (mean 3)")
ax.set_xlim(0, 12); ax.set_xlabel("interval length"); ax.set_title("Fig 4: length-biased sampling on one long Gamma(2,1) path")
ax.legend(); plt.show()
print(f"{len(gaps)} intervals on [0, {T_PATH}];  mean by index {gaps.mean():.4f} vs mu = {MU};  mean by time {L_U.mean():.4f} vs E[X^2]/mu = {EX2 / MU}")

### 4.5 결합분포: 같은 주변분포, 음의 상관 (Fig 5)

4.2의 $(A,R)$ 표본으로 결합 구조를 본다. 결합밀도 $f(a+r)/\mu$는 $a+r$에만 의존하므로 대각선 방향으로 등고선이 생기고, $\mathrm{Cov}(A,R)=-1/4$, $\mathrm{Corr}=-1/7$이다.

In [ ]:
n_scatter = min(3000, N_PATHS)
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(A[:n_scatter], R[:n_scatter], s=4, alpha=0.4)
ax.set_xlabel("age A(30)"); ax.set_ylabel("residual R(30)"); ax.set_xlim(0, 8); ax.set_ylim(0, 8)
ax.set_title("Fig 5: (age, residual) at t = 30, Gamma(2,1)"); plt.show()

cov_samples = (A - A.mean()) * (R - R.mean())
est_cov = mc_estimate(cov_samples)
print(f"Cov(A, R) = {est_cov}   (limit E[X^3]/(6 mu) - (E[X^2]/(2 mu))^2 = {EX3 / (6 * MU) - (EX2 / (2 * MU))**2})")
print(f"Corr(A, R) = {np.corrcoef(A, R)[0, 1]:.4f}   (limit -1/7 = {-1 / 7:.4f})")

조건부 함정 시연: 비조건부 상관은 음수지만, $L$이 큰 표본으로 조건을 주면 $A$와 $R$ 모두 커진다(둘 다 $L$의 조각이므로). "큰 $L$ ⇒ 큰 $A$"와 "$\mathrm{Cov}(A,R)<0$"은 서로 다른 조건 아래의 진술이라 모순이 아니다.

In [ ]:
q = np.quantile(L, [0, 0.25, 0.5, 0.75, 1.0])
lines = ["| L 사분위 | L 범위 | E[A \\| L in range] | E[R \\| L in range] |", "|---|---|---|---|"]
for k in range(4):
    m = (L >= q[k]) & (L <= q[k + 1])
    lines.append(f"| Q{k + 1} | [{q[k]:.2f}, {q[k + 1]:.2f}] | {A[m].mean():.3f} | {R[m].mean():.3f} |")
Markdown("\n".join(lines))

### 4.6 푸아송 특수 경우 (Fig 6)

비율 1인 푸아송 과정에서는 모든 유한 $t$에서 닫힌 형식이 있다: $E[A(t)]=1-e^{-t}$, $E[R(t)]=1$, $E[L(t)]=2-e^{-t}$. 경로마다 `poisson_arrivals`로 $[0,60]$의 도착을 만들고 여러 $t$에서 `age_residual_at`으로 읽는다($t=30$에서 다음 도착이 60 안에 없을 확률은 $e^{-30}$).

In [ ]:
N_POIS, T_GEN = scale(4_000), 60.0
t_list = [0.25, 0.5, 1.0, 2.0, 4.0, 30.0]
A_pois = {t: np.empty(N_POIS) for t in t_list}
R_pois = {t: np.empty(N_POIS) for t in t_list}
for i in range(N_POIS):
    arr = poisson_arrivals(1.0, T_GEN, rng)
    for t in t_list:
        A_pois[t][i], R_pois[t][i] = age_residual_at(arr, t)
for t in t_list:
    assert np.isfinite(R_pois[t]).all(), f"t={t}: some path had no arrival after t (increase T_GEN)"
L_pois = {t: A_pois[t] + R_pois[t] for t in t_list}
A_pois_05, L_pois_05 = A_pois[0.5], L_pois[0.5]
print(f"t=0.5: E[A] = {mc_estimate(A_pois_05)}  exact {1 - np.exp(-0.5):.4f}")
print(f"t=0.5: E[L] = {mc_estimate(L_pois_05)}  exact {2 - np.exp(-0.5):.4f}")

In [ ]:
tt = np.linspace(0.05, 40, 400)
fig, ax = plt.subplots(figsize=(9, 4))
for d, lab, exact in ((L_pois, "E[L(t)]", 2 - np.exp(-tt)), (A_pois, "E[A(t)]", 1 - np.exp(-tt)), (R_pois, "E[R(t)]", np.ones_like(tt))):
    ests = [mc_estimate(d[t]) for t in t_list]
    ax.errorbar(t_list, [e.mean for e in ests], yerr=[1.96 * e.se for e in ests], fmt="o", capsize=3, label=f"{lab} MC (95% CI)")
    ax.plot(tt, exact, lw=1, color=ax.lines[-1].get_color())
ax.axhline(EX2 / MU, color="k", ls="--", lw=1, label="Gamma(2,1) limit E[L] = 3")
ax.set_xscale("log"); ax.set_xlabel("t (log scale)"); ax.set_ylabel("mean")
ax.set_title("Fig 6: Poisson(1): E[L(t)] = 2 - e^{-t}, E[A(t)] = 1 - e^{-t}, E[R(t)] = 1")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0)); plt.show()

푸아송에서는 $E[R(t)]=1=\mu$가 모든 $t$에서 성립하고(무기억성), $E[L(t)]$는 $2=2\mu$로 수렴한다. 비율로 보면 $E[L]/\mu=1+\mathrm{CV}^2$: 푸아송(1)은 $\mathrm{CV}^2=1$이라 $2/1=2$, Gamma(2,1)은 $\mathrm{CV}^2=1/2$라 $3/2=1.5$. 검사 역설의 크기는 평균이 아니라 간격의 **변동성**이 정한다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 모두 공식으로 계산한다 (Gamma(2,1): $\mu=2$, $E[X^2]=6$, $E[X^3]=24$; 푸아송(1)은 $t=0.5$).

In [ ]:
exact = {
    "mean_age_limit": EX2 / (2 * MU),
    "mean_spread_limit": EX2 / MU,
    "P_age_gt_2": Fe_bar(2.0),
    "P_age_gt1_res_gt1": Fe_bar(1.0 + 1.0),
    "cov_age_residual": EX3 / (6 * MU) - (EX2 / (2 * MU)) ** 2,
    "E_spread_poisson_t05": 2 - np.exp(-0.5),
    "E_age_poisson_t05": 1 - np.exp(-0.5),
}
rows = [
    {"name": "E[A(30)] age, Gamma(2,1)", "predicted": predictions["mean_age_limit"], "estimate": est_A, "exact": exact["mean_age_limit"]},
    {"name": "E[R(30)] residual life", "predicted": predictions["mean_age_limit"], "estimate": est_R, "exact": exact["mean_age_limit"]},
    {"name": "E[L(30)] spread", "predicted": predictions["mean_spread_limit"], "estimate": est_L, "exact": exact["mean_spread_limit"]},
    {"name": "P(A > 2)", "predicted": predictions["P_age_gt_2"], "estimate": p_A2, "exact": exact["P_age_gt_2"]},
    {"name": "P(A > 1, R > 1)", "predicted": predictions["P_age_gt1_res_gt1"], "estimate": p_A1R1, "exact": exact["P_age_gt1_res_gt1"]},
    {"name": "Cov(A, R)", "predicted": predictions["cov_age_residual"], "estimate": est_cov, "exact": exact["cov_age_residual"]},
    {"name": "E[L(0.5)] Poisson(1)", "predicted": predictions["E_spread_poisson_t05"], "estimate": mc_estimate(L_pois_05), "exact": exact["E_spread_poisson_t05"]},
    {"name": "E[A(0.5)] Poisson(1)", "predicted": predictions["E_age_poisson_t05"], "estimate": mc_estimate(A_pois_05), "exact": exact["E_age_poisson_t05"]},
]
Markdown(compare_table(rows))

각 행이 어긋난다면:
- **E[A(30)], E[R(30)]** 이 1.5에서 벗어나면 정리 1의 $E[X^2]/(2\mu)$ 또는 헬퍼의 $S_{N(t)}$ 인덱싱이 틀린 것이다(둘 중 하나만 어긋나면 나이·잔여수명 대칭이 깨진 것).
- **E[L(30)]** 이 3이 아니라 2 근처면 "지표로 뽑기"를 한 것 — 검사 역설을 놓쳤다.
- **P(A > 2)** 가 $\bar F_e(2)$가 아니라 $\bar F(2)=0.406$ 근처면 극한분포가 $F$가 아니라 $F_e$임을 놓친 것이다.
- **P(A > 1, R > 1)** 이 0.3045 근처면 독립 가정이 (잘못) 들어간 것이고, $\bar F_e(2)=0.2707$이 맞다.
- **Cov(A, R)** 이 0 근처면 지수 간격을 쓴 것이거나 표본이 섞였다; $-0.25$가 맞다.
- **Poisson E[L(0.5)], E[A(0.5)]** 가 어긋나면 `age_residual_at`의 경계 처리($t$ 이전 도착이 없을 때 $A(t)=t$)를 의심한다.

In [ ]:
assert_close(est_A, exact["mean_age_limit"], k=4, name="E[age] Gamma(2,1)")
assert_close(est_R, exact["mean_age_limit"], k=4, name="E[residual] Gamma(2,1)")
assert_close(est_L, exact["mean_spread_limit"], k=4, name="E[spread] = E[X^2]/mu")
assert_close(p_A2, exact["P_age_gt_2"], k=4, name="P(A>2) = Fe_bar(2)")
assert_close(p_A1R1, exact["P_age_gt1_res_gt1"], k=4, name="P(A>1,R>1) = Fe_bar(2)")
assert_close(est_cov, exact["cov_age_residual"], k=4, name="Cov(A,R)")
assert_close(mc_estimate(L_pois_05), exact["E_spread_poisson_t05"], k=4, name="Poisson E[L(0.5)]")
assert_close(mc_estimate(A_pois_05), exact["E_age_poisson_t05"], k=4, name="Poisson E[A(0.5)]")
print("all assert_close passed (k=4)")

## 6. 연습문제

### E1 계산

간격 $X\sim$ Uniform(0,1). (a) $F_e(x)$를 구하라. (b) 극한 나이의 평균과 $P(A>1/2)$. (c) 극한 관찰 구간 길이의 평균 $E[L]$과 $\mu$의 차이를 $\sigma^2/\mu$로 확인하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$\mu=1/2$, $E[X^2]=1/3$, $\sigma^2=1/12$, $\bar F(y)=1-y$ ($0\le y\le1$).

(a) $F_e(x)=\frac1\mu\int_0^x(1-y)\,dy=2x-x^2$ ($0\le x\le1$), 밀도 $f_e(x)=2(1-x)$.

(b) $E[A]=E[X^2]/(2\mu)=\frac{1/3}{1}=\frac13$. $P(A>\tfrac12)=1-F_e(\tfrac12)=1-\tfrac34=\tfrac14$.

(c) $E[L]=E[X^2]/\mu=\frac23$. $E[L]-\mu=\frac23-\frac12=\frac16=\sigma^2/\mu=\frac{1/12}{1/2}$ ✓.

시뮬레이션으로 확인(간격이 짧으니 $t=30$은 $\mu$의 60배):

```python
Au, Ru = age_residual_many(lambda s, g: g.uniform(0, 1, s), 30.0, N_PATHS, 120, rng)   # 120개 합: 평균 60, sd 3.2 → t=30 미달 확률 무시
print(mc_estimate(Au), "vs 1/3;", mc_proportion(Au > 0.5), "vs 1/4;", mc_estimate(Au + Ru), "vs 2/3")
```

</details>

### E2 유도 후 시뮬레이션 검증

(a) 결합 극한밀도 $f(a+r)/\mu$에서 $L=A+R$의 밀도가 $\ell f(\ell)/\mu$임을 유도하라. (b) Gamma(2,1)에서 이것이 Gamma(3,1)임을 보이고 $E[L]$, $E[L^2]$, $\mathrm{Var}(L)$을 구하라. (c) 4.2의 `A, R`로 $E[L^2]$을 추정해 4 SE 안에 들어오는지 `assert_close`하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $(A,R)\mapsto(L,A)=(A+R,A)$는 야코비안 1인 변환이고 $L=\ell$일 때 $A$의 범위는 $[0,\ell]$이다. 따라서

$$P(L\in d\ell)=\Big(\int_0^\ell\frac{f(\ell)}\mu\,da\Big)d\ell=\frac{\ell f(\ell)}\mu\,d\ell.$$

$L$이 주어졌을 때 $A$는 $[0,L]$ 위 균등분포 — "구간 안의 위치는 균등"이라는 직관의 정확한 형태다.

(b) $\ell\cdot\ell e^{-\ell}/2=\ell^2e^{-\ell}/\Gamma(3)$ = Gamma(3,1). $E[L]=3$, $E[L^2]=3\cdot4=12$ ($=E[X^3]/\mu=24/2$), $\mathrm{Var}(L)=3$.

(c) $L^2$의 sd는 $\sqrt{E[L^4]-144}=\sqrt{360-144}\approx14.7$ (Gamma(3,1): $E[L^4]=3\cdot4\cdot5\cdot6=360$)이라 4만 경로면 SE $\approx0.07$.

```python
est_L2 = mc_estimate(L**2)
print(est_L2, "vs E[X^3]/mu =", EX3 / MU)
assert_close(est_L2, 12.0, k=4, name="E[L^2] = Gamma(3,1) second moment")
```

</details>

### E3 가정을 깨보기

간격을 상수 $X\equiv2$(격자, $d=2$)로 바꾸어라(`sampler = lambda n, g: np.full(n, 2.0)`). (a) $t=30$과 $t=30.5$에서 여러 경로의 $A(t)$ 분포를 구하라 — 무엇이 이상한가? (b) 한 경로에서 $[0,T]$ 위 촘촘한 격자로 시간평균 $\frac1T\int_0^TA(s)\,ds$와 $\frac1T\int_0^T\mathbf 1\{A(s)\le1\}\,ds$를 계산하라. (c) 정리 1과 정리 2 중 어느 것이 살아남는가? $E[L]$은?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $A(30)\equiv0$, $A(30.5)\equiv0.5$: 경로에 무관한 결정값이고, $t$를 0.5만 옮겨도 분포가 완전히 바뀐다. $A(t)=t\bmod2$이므로 $t\to\infty$ 극한분포가 없다 — 격자 신호.

(b) 시간평균 나이 $\to1=E[X^2]/(2\mu)=4/4$, $P_{\text{time}}(A\le1)=1/2=F_e(1)$ ($F_e$ = Uniform(0,2), $f_e(x)=\bar F(x)/\mu=\mathbf 1\{x<2\}/2$).

(c) 정리 1(시간평균, 재생-보상)은 격자 조건이 없으므로 살아남고, 정리 2(극한분포)는 비격자 가정이 깨져 무너진다. $E[L]=E[X^2]/\mu=4/2=2=\mu$: $\sigma^2=0$이라 검사 역설이 사라진다(모든 간격이 같으니 길이 편향이 있을 수 없다). 비교: Gamma(2,1)은 비격자라 두 정리 모두 성립.

```python
const_shape = lambda shape, g: np.full(shape, 2.0)
for t in (30.0, 30.5):
    A3, R3 = age_residual_many(const_shape, t, 200, 20, rng)
    print(f"t={t}: A takes values {np.unique(A3)}, R takes values {np.unique(R3)}")
T = 1000.0
times_c = renewal_times(lambda n, g: np.full(n, 2.0), T, rng)
s = np.linspace(0, T, 200_001)
A_s, R_s = age_residual_process(times_c, s)
print("time-average age", A_s.mean(), "vs 1;  time fraction A<=1", (A_s <= 1).mean(), "vs 1/2")
```

</details>

## 7. 정리

1. $A(t)=t-S_{N(t)}$, $R(t)=S_{N(t)+1}-t$, $L=A+R$은 시각 $t$를 덮는 간격 $X_{N(t)+1}$의 세 조각이다.
2. 재생-보상으로 시간평균 분포는 항상 $F_e(x)=\frac1\mu\int_0^x\bar F$, 평균 $E[X^2]/(2\mu)=\frac\mu2(1+\mathrm{CV}^2)$; 비격자면 $t\to\infty$ 극한분포도 같다.
3. 결합 극한 $P(A>x,R>y)=\bar F_e(x+y)$: 주변분포는 같고 독립은 아니다(Gamma(2,1): Cov $=-1/4$, Corr $=-1/7$); 지수분포에서만 독립.
4. 검사 역설: 시각으로 뽑은 간격은 길이 편향 밀도 $xf(x)/\mu$를 따르고 평균 $E[X^2]/\mu=\mu+\sigma^2/\mu\ge\mu$. Gamma(2,1)이면 Gamma(3,1): 3 vs 2.
5. 푸아송이면 $F_e=F$, $E[L(t)]=(2-e^{-\lambda t})/\lambda$ (02d): 버스 대기는 $1/\lambda$이지 $1/(2\lambda)$가 아니다.

**Trap 카드.** Q: 버스 간격이 평균 10분(어떤 분포든)일 때 무작위로 도착한 승객의 평균 대기시간은 5분인가?
→ A: 아니다. $E[R]=\frac{E[X^2]}{2\mu}=5(1+\mathrm{CV}^2)$분. 정확히 10분 간격이면 5분, 지수분포면 10분, 더 불규칙하면 10분 이상. "평균의 절반"은 상수 간격에서만 맞다.

**다음 노트북: 04c.** 마르코프 사슬에서 한 상태로의 복귀 시각들이 재생 과정을 이룬다는 사실(강마르코프)에 04a의 재생-보상 정리를 적용하면 01d에서 미뤄 둔 $\pi(x)=1/E_x[T_x]$가 증명되고, 시간평균(에르고딕 정리)이 왜 비주기성 없이도 성립하는지가 이 노트북의 "시간평균 vs 극한분포" 구분(격자 = 주기)으로 이해된다.

`log/progress.md` 한 줄 템플릿: `| 날짜 | 04b 나이·잔여수명·검사 역설 | 예측 7개 중 __개 적중 | 막힌 곳: __ | 다음: 04c |`